# Linked Lists


## Topic overview

Sequential pointer chains; most patterns are pointer manipulations.

## Pattern-recognition rules

- Reverse in place.
- Detect cycles (Floyd's).
- Merge sorted lists.
- Dummy-head to simplify insertion.

## Common data structures

- Singly linked
- Doubly linked

## Standard complexity expectations

- Almost always O(n) time, O(1) extra space.

## Common mistakes

- Losing the head pointer.
- Not handling the empty list.

## Original illustrative example

In [ ]:
# Replace with an ORIGINAL example. Do not paste external
# problem statements. See src/algorithms/ for reusable helpers.
example_input = []
example_expected = None

## Add solved problems below

Each new sub-section should follow the template in `../templates/notebook_template.ipynb`.

1. [Reverse Linked List](#reverse-linked-list)
2. [Merge Two Sorted Linked Lists](#merge-two-sorted-linked-lists)


# Reverse Linked List

## Metadata

- Source: NeetCode / LeetCode 206
- Problem URL: https://leetcode.com/problems/reverse-linked-list/
- Difficulty: Easy
- Topic: Linked Lists
- Language: C++ (Python below is the same pointer walk, so this notebook can run it)
- Date started: 2026-10-04
- Date solved: 2026-10-04
- Current mastery level: 1
- Last reviewed: 2026-10-04
- Next review:


## Problem statement in my own words

You are given the head of a singly linked list. Reverse the links and
return the new head.

```
head
 ↓
[0] → [1] → [2] → [3] → nullptr

becomes

                new head
                   ↓
nullptr ← [0] ← [1] ← [2] ← [3]
```

The nodes stay where they are. Only each `next` pointer changes direction.
An empty list stays empty.


## Inputs, outputs, and constraints

- Input: `ListNode* head`
- Output: `ListNode*` — the new beginning
- Constraints:
  - $0 \le n \le 1000$
  - $-1000 \le$ `Node.val` $\le 1000$

NeetCode's node is a pointer type:

```cpp
struct ListNode {
    int val;
    ListNode* next;

    ListNode(int val) {
        this->val = val;
        next = nullptr;
    }
};
```

In C++, a pointer stores an address. `next` is not the next value. It is
the address of the next node, or `nullptr` when the chain ends.


## Examples

| Input | Expected | Notes |
|---|---|---|
| `head = [0, 1, 2, 3]` | `[3, 2, 1, 0]` | every link flips |
| `head = []` | `[]` | `head` is already `nullptr` |
| `head = [7]` | `[7]` | one node, `next` was already null |


## The danger

Suppose the walk has already reversed `0`, and `curr` is `1`:

```
prev       curr
 ↓           ↓
[0]         [1] → [2] → [3] → nullptr
```

If you write `curr->next = prev` immediately, node `1` now points at
`0`, and the address of `2` is gone. The rest of the list is leaked.

Save the forward link first.

```cpp
ListNode* prev = nullptr;  // reversed portion
ListNode* curr = head;     // node being processed
ListNode* next = nullptr;  // saved successor
```

| Pointer | Job |
|---|---|
| `prev` | head of the already-reversed prefix |
| `curr` | node we are about to flip |
| `next` | where `curr` used to point, so the walk can continue |


## The four moves

At every node, in this order:

1. Save `curr->next`
2. Reverse `curr->next` so it points at `prev`
3. Move `prev` onto `curr`
4. Move `curr` onto the saved node

```text
prev = nullptr
curr = head

while curr exists:
    next = curr.next
    curr.next = prev
    prev = curr
    curr = next

return prev
```

Return `prev`, not `curr`. When the loop stops, `curr` is `nullptr` and
`prev` is the old tail, which is the new head.

## Trace — `[0, 1, 2, 3]`

Start: `prev = nullptr`, `curr = 0`.

| Step | save | after reverse | then |
|---|---|---|---|
| 1 | `next = 1` | `nullptr ← [0]` | `prev = 0`, `curr = 1` |
| 2 | `next = 2` | `[0] ← [1]` | `prev = 1`, `curr = 2` |
| 3 | `next = 3` | `[1] → [0] → nullptr`, `prev = 2` | `curr = 3` |
| 4 | `next = nullptr` | `[3] → [2] → [1] → [0] → nullptr` | `curr = nullptr` |

The loop invariant, checked before each iteration:

> `prev` is a correctly reversed list of every node already processed.
> `curr` is the first node not yet processed.

Midway:

```
REVERSED                 NOT YET
[1] → [0] → nullptr      [2] → [3] → nullptr
 ↑                        ↑
prev                     curr
```

One iteration pulls `2` off the unprocessed side and puts it at the
front of the reversed side. When `curr` becomes `nullptr`, `prev` is
the whole list.


## C++ solution

```cpp
class Solution {
public:
    ListNode* reverseList(ListNode* head) {
        // Already-reversed portion. Empty at the start.
        ListNode* prev = nullptr;

        // Node we are about to flip.
        ListNode* curr = head;

        while (curr != nullptr) {
            // 1. Save the forward link before it is overwritten.
            ListNode* next = curr->next;

            // 2. Point this node at the reversed prefix.
            curr->next = prev;

            // 3. The reversed prefix now starts here.
            prev = curr;

            // 4. Continue from the saved successor.
            curr = next;
        }

        // prev is the old tail, which is the new head.
        return prev;
    }
};
```

Do not memorize the function. Memorize the four lines:

```cpp
ListNode* next = curr->next;  // save
curr->next = prev;            // reverse
prev = curr;                  // advance prev
curr = next;                  // advance curr
```

Save, reverse, advance, advance. The same four moves show up inside
Reverse Linked List II, Reverse Nodes in k-Group, and Reorder List.


## Complexity

Let $n$ be the number of nodes. Each node is visited once.

$$
T(n) = O(n)
$$

Only three pointers are stored, however long the list is. That beats
the recursive reversal, which uses $O(n)$ call-stack space.

$$
S(n) = O(1)
$$

## Edge cases

- Empty list: `curr` starts as `nullptr`, the loop never runs, and
  `prev` is returned as `nullptr`. No extra `if`.
- One node: save `nullptr`, point `next` at `nullptr`, return that node.
- Two nodes: one real swap of the single link.
- The values never matter. Only addresses move.


## Testing

The checks use the same four assignments as the C++ loop. This kernel
is Python, so the mirror is what actually runs.


In [ ]:
class ListNode:
    def __init__(self, val: int = 0, next: "ListNode | None" = None) -> None:
        self.val = val
        self.next = next


def from_list(values: list[int]) -> ListNode | None:
    dummy = ListNode(0)
    tail = dummy
    for value in values:
        tail.next = ListNode(value)
        tail = tail.next
    return dummy.next


def to_list(head: ListNode | None) -> list[int]:
    out: list[int] = []
    while head is not None:
        out.append(head.val)
        head = head.next
    return out


def reverse_list(head: ListNode | None) -> ListNode | None:
    prev = None
    curr = head
    while curr is not None:
        nxt = curr.next
        curr.next = prev
        prev = curr
        curr = nxt
    return prev


assert to_list(reverse_list(from_list([0, 1, 2, 3]))) == [3, 2, 1, 0]
assert to_list(reverse_list(from_list([]))) == []
assert to_list(reverse_list(from_list([7]))) == [7]
assert to_list(reverse_list(from_list([1, 2]))) == [2, 1]

print("Reverse Linked List checks passed")


## Visualization

Nodes stay in their original places. Only the arrows flip. That is the
point of the algorithm: we are not moving values into a new list.

- orange: `curr`, the node just reversed in this frame
- gold: the saved `next`
- navy: `prev`
- green: the rest of the already-reversed chain
- gray: `nullptr`

If `ipywidgets` is installed, use **Next step** / **Prev** / **Reset**.
Otherwise every iteration is drawn as its own frame.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, List, Optional

import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch


@dataclass(frozen=True)
class ReverseFrame:
    step: int
    values: List[int]
    nxt: Dict[int, Optional[int]]
    prev: Optional[int]
    curr: Optional[int]
    saved: Optional[int]
    decision: str


def reverse_frames(values: List[int]) -> List[ReverseFrame]:
    """Record each save / reverse / advance, matching the C++ loop."""
    n = len(values)
    nxt: Dict[int, Optional[int]] = {i: (i + 1 if i + 1 < n else None) for i in range(n)}
    prev: Optional[int] = None
    curr: Optional[int] = 0 if n else None
    frames: List[ReverseFrame] = [
        ReverseFrame(0, list(values), dict(nxt), prev, curr, None, "start: prev = nullptr, curr = head")
    ]
    if n == 0:
        return frames
    step = 1
    while curr is not None:
        saved = nxt[curr]
        nxt[curr] = prev
        frames.append(
            ReverseFrame(
                step,
                list(values),
                dict(nxt),
                prev,
                curr,
                saved,
                f"save next, reverse node {values[curr]}, then advance",
            )
        )
        prev = curr
        curr = saved
        step += 1
    frames.append(
        ReverseFrame(
            step,
            list(values),
            dict(nxt),
            prev,
            curr,
            None,
            "curr is nullptr — return prev (the new head)",
        )
    )
    return frames


def _node_color(idx: int, frame: ReverseFrame) -> str:
    if frame.curr == idx:
        return "#e76f51"
    if frame.saved == idx and frame.curr is not None:
        return "#e9c46a"
    if frame.prev == idx:
        return "#1d3557"
    # Nodes already reversed sit on the chain that starts at prev.
    seen = set()
    walk = frame.prev
    while walk is not None and walk not in seen:
        seen.add(walk)
        walk = frame.nxt[walk]
    if idx in seen:
        return "#2a9d8f"
    return "#4c78a8"


def plot_reverse_frame(frame: ReverseFrame, *, ax=None):
    """Nodes stay put. Only the next arrows change direction."""
    created = ax is None
    values = frame.values
    n = len(values)
    if created:
        _, ax = plt.subplots(figsize=(max(8, (n + 2) * 1.7), 3.6))

    ax.set_xlim(-2.1, max(n, 1) * 1.7 + 1.6)
    ax.set_ylim(-1.5, 2.2)
    ax.axis("off")
    ax.set_title(f"Step {frame.step}   {frame.decision}", loc="left", fontsize=11)

    if n == 0:
        ax.text(0, 0.4, "head = nullptr    return nullptr", fontsize=13, color="#1d3557")
        return ax

    xs = [i * 1.7 for i in range(n)]

    def box(x: float, y: float, label: str, face: str, text_color: str = "white") -> None:
        ax.add_patch(
            FancyBboxPatch(
                (x - 0.42, y - 0.32),
                0.84,
                0.64,
                boxstyle="round,pad=0.02,rounding_size=0.08",
                facecolor=face,
                edgecolor="#1d3557",
                linewidth=1.2,
            )
        )
        ax.text(x, y, label, ha="center", va="center", fontsize=12, fontweight="bold", color=text_color)

    box(-1.3, 0.4, "null", "#6c757d")
    for i, value in enumerate(values):
        face = _node_color(i, frame)
        text_color = "#1d3557" if face == "#e9c46a" else "white"
        box(xs[i], 0.4, str(value), face, text_color)
        ax.text(xs[i], -0.15, f"[{i}]", ha="center", va="top", fontsize=8, color="#6b6b6b")

    for i in range(n):
        target = frame.nxt[i]
        x0 = xs[i]
        if target is None:
            # Node 0's null is the left sentinel. Any other null is the
            # still-unreversed tail, drawn to the right of the last node.
            if i == 0:
                start = (x0 - 0.46, 0.4)
                end = (-1.3 + 0.48, 0.4)
            else:
                x1 = xs[-1] + 1.35
                box(x1, 0.4, "null", "#6c757d")
                start = (x0 + 0.46, 0.4)
                end = (x1 - 0.48, 0.4)
            ax.add_patch(
                FancyArrowPatch(
                    start,
                    end,
                    arrowstyle="-|>",
                    mutation_scale=12,
                    color="#1d3557",
                    linewidth=1.3,
                    connectionstyle="arc3,rad=0.0",
                )
            )
        else:
            x1 = xs[target]
            backward = target < i
            rad = 0.28 if backward else -0.22
            y = 0.78 if backward else 0.4
            ax.add_patch(
                FancyArrowPatch(
                    (x0 + (-0.46 if backward else 0.46), y),
                    (x1 + (0.46 if backward else -0.46), y),
                    arrowstyle="-|>",
                    mutation_scale=12,
                    color="#1b6b62" if backward else "#1d3557",
                    linewidth=1.4,
                    connectionstyle=f"arc3,rad={rad}",
                )
            )

    def tag(idx: Optional[int], name: str, y: float, color: str) -> None:
        if idx is None:
            if name == "prev":
                ax.text(-1.3, 1.15, name, ha="center", va="bottom", fontsize=10, fontweight="bold", color=color)
            else:
                ax.text(
                    xs[-1] + 1.15,
                    1.55,
                    f"{name} = nullptr",
                    ha="left",
                    fontsize=10,
                    fontweight="bold",
                    color=color,
                )
            return
        ax.annotate(
            name,
            xy=(xs[idx], 0.76),
            xytext=(xs[idx], y),
            ha="center",
            fontsize=10,
            fontweight="bold",
            color=color,
            arrowprops={"arrowstyle": "->", "color": color, "lw": 1.0},
        )

    tag(frame.prev, "prev", 1.7, "#1d3557")
    tag(frame.curr, "curr", 1.35, "#9b2226")
    if frame.saved is not None and frame.curr is not None:
        tag(frame.saved, "next", 1.95, "#b08900")

    if created:
        plt.tight_layout()
    return ax


def visualize_reverse_all(values: List[int]) -> None:
    frames = reverse_frames(values)
    print(f"values={values}")
    for frame in frames:
        prev = "null" if frame.prev is None else frame.values[frame.prev]
        curr = "null" if frame.curr is None else frame.values[frame.curr]
        print(f"step {frame.step}: prev={prev} curr={curr}  {frame.decision}")
        plot_reverse_frame(frame)
    plt.show()


def step_through_reverse(values: List[int]) -> None:
    frames = reverse_frames(values)
    try:
        import ipywidgets as widgets
        from IPython.display import clear_output, display
    except ImportError:
        visualize_reverse_all(values)
        return

    output = widgets.Output()
    state = {"i": 0}

    def render() -> None:
        with output:
            clear_output(wait=True)
            frame = frames[state["i"]]
            print(f"frame {state['i'] + 1}/{len(frames)}")
            plot_reverse_frame(frame)
            plt.show()

    def on_next(_=None) -> None:
        state["i"] = min(state["i"] + 1, len(frames) - 1)
        render()

    def on_prev(_=None) -> None:
        state["i"] = max(state["i"] - 1, 0)
        render()

    def on_reset(_=None) -> None:
        state["i"] = 0
        render()

    prev_btn = widgets.Button(description="Prev")
    next_btn = widgets.Button(description="Next step")
    reset_btn = widgets.Button(description="Reset")
    prev_btn.on_click(on_prev)
    next_btn.on_click(on_next)
    reset_btn.on_click(on_reset)
    display(widgets.HBox([prev_btn, next_btn, reset_btn]), output)
    render()

print("Example — reverse [0, 1, 2, 3]")
step_through_reverse([0, 1, 2, 3])

print("\nEmpty list")
step_through_reverse([])


## Alternative approaches

- Recursion: reverse the tail, then point `head->next->next = head`
  and `head->next = nullptr`. Same $O(n)$ time, $O(n)$ stack.
- Copy values into a vector, reverse the vector, write them back.
  $O(n)$ extra memory, and it hides the pointer lesson.
- Stack of node pointers: also $O(n)$ extra memory.

## Mistakes I made

- Reversing `curr->next` before saving it, which drops the rest of the list.
- Returning `curr` or `head`. Both are wrong at the end: `curr` is null,
  and the original `head` is now the tail.
- Special-casing the empty list. The loop already handles it.
- Writing `next` as a name that shadows something, then using the field
  `curr->next` after it was overwritten. The local copy has to be a
  different pointer.

## Pattern recognition

Singly linked reversal is always:

```cpp
ListNode* next = curr->next;
curr->next = prev;
prev = curr;
curr = next;
```

If a later problem reverses only part of a list, the same four lines
run between two fixed endpoints. The endpoints are what change, not
the flip.

## Related problems

- Reverse Linked List II — reverse one index range, then stitch it back
- Reverse Nodes in k-Group — this loop once per block of $k$
- Swap Nodes in Pairs — reversal with $k = 2$
- Reorder List — reverse the second half, then merge
- Palindrome Linked List — reverse half, then compare

## Real-world or engineering connection

Any singly linked buffer (free lists, intrusive lists, packet chains)
is reversed by retargeting `next`, not by sliding the payloads. Losing
the successor pointer is a use-after-free waiting to happen: you still
need that address to walk forward.

## Final takeaways

1. A `next` pointer is an address, not a value.
2. Save that address before you overwrite it.
3. Four moves: save, reverse, advance `prev`, advance `curr`.
4. Return `prev`.


## Reattempt log

| Date | Mastery | Notes |
|---|---|---|
| 2026-10-04 | 1 | First write-up: iterative C++ reversal, save before reverse |


# Merge Two Sorted Linked Lists

## Metadata

- Source: NeetCode / LeetCode 21
- Problem URL: https://neetcode.io/problems/merge-two-sorted-linked-lists
- Difficulty: Easy
- Topic: Linked Lists
- Language: C++ (Python below is the same pointer walk, so this notebook can run it)
- Date started: 2026-10-06
- Date solved: 2026-10-06
- Current mastery level: 1
- Last reviewed: 2026-10-06
- Next review: 2026-10-07


## Problem statement in my own words

You are given the heads of two **already-sorted** singly linked lists.
Rewire their nodes into one sorted list and return the new head.

```
list1
  ↓
 [1] → [2] → [4] → nullptr

list2
  ↓
 [1] → [3] → [5] → nullptr

becomes

 [1] → [1] → [2] → [3] → [4] → [5] → nullptr
  ↑
 returned
```

The nodes stay where they are. Only `next` pointers change. Do not
allocate a fresh node per value. An empty input is just `nullptr`.


## Inputs, outputs, and constraints

- Inputs: `ListNode* list1`, `ListNode* list2` — heads of two sorted lists
- Output: `ListNode*` — head of the merged sorted list
- Constraints:
  - $0 \le n, m \le 100$
  - $-100 \le$ `Node.val` $\le 100$
  - both lists are already sorted non-decreasing

NeetCode's node is the same pointer type as Reverse Linked List:

```cpp
struct ListNode {
    int val;
    ListNode* next;

    ListNode(int val) {
        this->val = val;
        next = nullptr;
    }
};
```

`list1` is not "the list." It is the address of the first unprocessed
node. `list1 = list1->next` does not copy a value. It walks the pointer
forward.


## Examples

| Input | Expected | Notes |
|---|---|---|
| `list1 = [1, 2, 4]`, `list2 = [1, 3, 5]` | `[1, 1, 2, 3, 4, 5]` | equal heads: take `list1` first |
| `list1 = []`, `list2 = [1, 2]` | `[1, 2]` | loop never runs; splice `list2` |
| `list1 = []`, `list2 = []` | `[]` | splice `nullptr`, return `nullptr` |


## Setup

Naive approach: walk both lists into a vector, sort, allocate new nodes.
That is $O((n+m)\log(n+m))$ time and $O(n+m)$ new nodes. It also ignores
the two facts that make the problem interesting:

1. both inputs are already sorted
2. the answer must be made of the **existing** nodes

The constraint that kills the naive rebuild: we are not allowed to treat
the lists as arrays of values. We have to change links.

## Governing principle

For two sorted sequences, the next merged node is whichever remaining
head is smaller.

If `list1->val <= list2->val`, `list1`'s current node comes next.
Otherwise `list2`'s current node comes next.

Think of two fingers. At every step, only the two fronts are candidates.
Everything behind a finger is already larger, so it can wait.


## The annoying part: where does the merged list start?

Special-casing the first node works and is messy:

```cpp
ListNode* head = nullptr;
// then, on every attach: if head is null, set it, else set tail->next
```

The dummy node removes that branch. Its value does not matter.

```
dummy
  ↓
 [-1]
  ↑
 tail
```

`tail` always points at the last node currently in the merged list.
After the first attach:

```
dummy → [1]
         ↑
        tail
```

At the end we return `dummy.next`, not `dummy`. The dummy was never
part of either input, so it is not part of the answer.

This is the same dummy-head trick as the `LinkedList` class under
`Data Structures & Algorithms/singlyLinkedList/`: a fake node sitting
in front of the real chain so insertion at the front is not a special
case.


## Algorithm design

Four pointers:

| Pointer | Job |
|---|---|
| `list1` | first unprocessed node of the first list |
| `list2` | first unprocessed node of the second list |
| `dummy` | temporary node before the merged list |
| `tail` | last node currently on the merged list |

While both lists still have nodes, attach the smaller front, walk that
list forward, then walk `tail` onto the node you just attached.

When one list goes empty, the other is already sorted. Hang it off
`tail->next` in one assignment. Do **not** walk `tail` after that splice.
The function is about to return.

Three pointer operations on every loop iteration:

1. Attach a node (`tail->next = ...`)
2. Advance the list that node came from
3. Advance `tail`

Skip any of the three and the chain either loses a suffix or `tail`
stops pointing at the end.

```text
create dummy node
tail = dummy

while list1 exists AND list2 exists:
    if list1.value <= list2.value:
        tail.next = list1
        list1 = list1.next
    else:
        tail.next = list2
        list2 = list2.next
    tail = tail.next

attach whichever list still has nodes
return dummy.next
```


## C++ solution

```cpp
class Solution {
public:
    ListNode* mergeTwoLists(ListNode* list1, ListNode* list2) {
        // Temporary node placed before the actual merged list.
        ListNode dummy;

        // tail always points to the last node in the merged list.
        ListNode* tail = &dummy;

        // Continue while both lists still contain nodes.
        while (list1 != nullptr && list2 != nullptr) {
            // Choose the smaller current node.
            if (list1->val <= list2->val) {
                // Attach list1's current node.
                tail->next = list1;
                // Move list1 forward.
                list1 = list1->next;
            } else {
                // Attach list2's current node.
                tail->next = list2;
                // Move list2 forward.
                list2 = list2->next;
            }
            // Move tail to the node we just attached.
            tail = tail->next;
        }

        // At least one list is now empty.
        // Attach whichever list still contains nodes.
        if (list1 != nullptr) {
            tail->next = list1;
        } else {
            tail->next = list2;
        }

        // dummy itself is not part of the answer.
        return dummy.next;
    }
};
```

`ListNode dummy;` is a stack object. `ListNode* tail = &dummy;` stores
its address. We never `new` the dummy, so we never `delete` it.

Do not memorize the function. Memorize the dummy-head pattern:

```cpp
ListNode dummy;
ListNode* tail = &dummy;

while (/* nodes available */) {
    tail->next = /* chosen node */;
    /* source */ = /* source */->next;
    tail = tail->next;
}

return dummy.next;
```


## Trace — `[1, 2, 4]` and `[1, 3, 5]`

Start: both fingers on the first `1`. `tail` sits on `dummy`.

| Step | Compare | Take | Why | Merged so far | `list1` | `list2` |
|---|---|---|---|---|---|---|
| 1 | $1 \le 1$ | `list1` | equal heads use `<=`, so `list1` wins | `dummy → 1` | `2 → 4` | `1 → 3 → 5` |
| 2 | $1 < 2$ | `list2` | remaining `list2` head is smaller | `dummy → 1 → 1` | `2 → 4` | `3 → 5` |
| 3 | $2 \le 3$ | `list1` | 2 is the next overall minimum | `dummy → 1 → 1 → 2` | `4` | `3 → 5` |
| 4 | $3 < 4$ | `list2` | 3 is smaller | `dummy → 1 → 1 → 2 → 3` | `4` | `5` |
| 5 | $4 \le 5$ | `list1` | 4 is smaller | `dummy → 1 → 1 → 2 → 3 → 4` | `nullptr` | `5` |

The `while` stops because `list1` is `nullptr`. One assignment finishes
the job:

```cpp
tail->next = list2;  // hang 5 → nullptr off the 4
```

`tail` is **not** walked onto `5`. We are done. Return `dummy.next`:

$$
[1, 1, 2, 3, 4, 5]
$$

The loop invariant, checked before each iteration:

> The chain `dummy.next … tail` is a correctly merged prefix of every
> node already processed. `list1` and `list2` are the sorted remainders.


## Why `->` appears everywhere

`ListNode* list1` is a pointer. The node lives at that address.

| Expression | Meaning |
|---|---|
| `list1->val` | the integer stored in that node |
| `list1->next` | the address of the successor, or `nullptr` |
| `(*list1).val` | the same as `list1->val`; nobody writes this |

## Why `tail->next = list1` is the whole problem

That line does not copy a value. We are not doing
`tail->next->val = list1->val`. We are changing the pointer itself.

Before:

```
tail
 ↓
[1] → nullptr

list1
 ↓
[2] → [4] → nullptr
```

After `tail->next = list1`:

```
tail
 ↓
[1] ─────┐
         ↓
        [2] → [4] → nullptr
         ↑
       list1
```

The existing nodes are now one chain. That is why the problem says the
new list is made of nodes from `list1` and `list2`. We rewired the
graph. We did not rebuild the data.

Then `list1 = list1->next` walks the **source** finger onto `4`, and
`tail = tail->next` walks the **builder** finger onto `2`. Same node,
two jobs, two pointers.


## Complexity analysis

Let $n$ be the length of `list1` and $m$ the length of `list2`.

| Work | Cost | Why |
|---|---|---|
| each `while` iteration | $O(1)$ | one compare, three pointer writes |
| nodes processed in the loop | $O(\min(n, m))$ | one list empties first |
| leftover splice | $O(1)$ | a single `tail->next` assignment |
| **total time** | $O(n + m)$ | every node is looked at at most once |

$$
T(n, m) = O(n + m)
$$

Space is a dummy on the stack plus a handful of pointers. No new nodes
for the $n + m$ values, and no recursion.

$$
S(n, m) = O(1)
$$

## Edge-case rationale

The tempting shortcut is "the loop handles everything, so skip the
splice." That breaks the moment one list is empty, including the
official empty-input cases.

| Case | What happens | Why the shortcut fails |
|---|---|---|
| both empty | loop skips; `tail->next = list2` is `nullptr` | there is nothing to compare |
| `list1` empty, `list2 = [1, 2]` | loop skips; splice `list2` | the whole answer is the leftover |
| `list2` empty, `list1` nonempty | splice `list1` | same, other side |
| all of `list1` smaller | loop consumes `list1`; splice `list2` | leftover is the entire second list |
| equal heads | `<=` takes `list1` | a strict `<` still works, but this is the tie-break we chose |

Empty lists do not need an extra `if (list1 == nullptr) return list2`
guard. The dummy plus the splice already returns the other list, or
`nullptr` when both are empty.


## Testing

The checks use the same dummy / tail / three-move loop as the C++. This
kernel is Python, so the mirror is what actually runs.


In [ ]:
class ListNode:
    def __init__(self, val: int = 0, next: "ListNode | None" = None) -> None:  # noqa: UP037
        self.val = val
        self.next = next


def from_list(values: list[int]) -> ListNode | None:
    dummy = ListNode(0)
    tail = dummy
    for value in values:
        tail.next = ListNode(value)
        tail = tail.next
    return dummy.next


def to_list(head: ListNode | None) -> list[int]:
    out: list[int] = []
    while head is not None:
        out.append(head.val)
        head = head.next
    return out


def merge_two_lists(
    list1: ListNode | None, list2: ListNode | None
) -> ListNode | None:
    """Rewire two sorted lists into one. Existing nodes, no new value copies."""
    dummy = ListNode(0)
    # tail is the last node of the merged prefix. Dummy at the start so
    # the first attach is not a special case.
    tail = dummy
    while list1 is not None and list2 is not None:
        # Next merged node is whichever remaining head is smaller.
        if list1.val <= list2.val:
            tail.next = list1
            list1 = list1.next
        else:
            tail.next = list2
            list2 = list2.next
        # Walk the builder onto the node we just hung on the chain.
        tail = tail.next
    # One list is empty. The other is already sorted — attach it whole.
    tail.next = list1 if list1 is not None else list2
    return dummy.next


assert to_list(merge_two_lists(from_list([1, 2, 4]), from_list([1, 3, 5]))) == [
    1,
    1,
    2,
    3,
    4,
    5,
]
assert to_list(merge_two_lists(from_list([]), from_list([1, 2]))) == [1, 2]
assert to_list(merge_two_lists(from_list([]), from_list([]))) == []
assert to_list(merge_two_lists(from_list([1, 2]), from_list([]))) == [1, 2]
assert to_list(merge_two_lists(from_list([1, 2]), from_list([5, 6]))) == [1, 2, 5, 6]
assert to_list(merge_two_lists(from_list([5]), from_list([1]))) == [1, 5]

print("Merge Two Sorted Linked Lists checks passed")


## Visualization

Nodes stay in their original lists until a pointer rewrite pulls them
onto the dummy-headed chain. That is the point of the algorithm: we are
not copying values into a new list.

- blue: remaining `list1`
- teal: remaining `list2`
- gray: dummy / `nullptr`
- green: already-merged prefix
- orange: node just attached (or last node of a leftover splice)
- red `tail`: last node the loop has claimed — it does **not** move
  during the final splice

If `ipywidgets` is installed, use **Next step** / **Prev** / **Reset**.
Otherwise every iteration is drawn as its own frame.


In [ ]:
from src.visualizations.linked_lists import (
    merge_frames,
    plot_merge_frame,
    visualize_merge_all,
)


def step_through_merge(list1: list[int], list2: list[int]) -> None:
    frames = merge_frames(list1, list2)
    try:
        import ipywidgets as widgets
        from IPython.display import clear_output, display
    except ImportError:
        visualize_merge_all(list1, list2)
        return

    output = widgets.Output()
    state = {"i": 0}

    def render() -> None:
        with output:
            clear_output(wait=True)
            frame = frames[state["i"]]
            print(f"frame {state['i'] + 1}/{len(frames)}")
            plot_merge_frame(frame)
            import matplotlib.pyplot as plt

            plt.show()

    def on_next(_=None) -> None:
        state["i"] = min(state["i"] + 1, len(frames) - 1)
        render()

    def on_prev(_=None) -> None:
        state["i"] = max(state["i"] - 1, 0)
        render()

    def on_reset(_=None) -> None:
        state["i"] = 0
        render()

    prev_btn = widgets.Button(description="Prev")
    next_btn = widgets.Button(description="Next step")
    reset_btn = widgets.Button(description="Reset")
    prev_btn.on_click(on_prev)
    next_btn.on_click(on_next)
    reset_btn.on_click(on_reset)
    display(widgets.HBox([prev_btn, next_btn, reset_btn]), output)
    render()


print("Example — merge [1, 2, 4] and [1, 3, 5]")
step_through_merge([1, 2, 4], [1, 3, 5])

print("\nOne list empty")
step_through_merge([], [1, 2])

print("\nBoth empty")
step_through_merge([], [])


## Alternative approaches

Recursive rewrite: the next head is the smaller node, and its `next` is
the merge of whatever remains.

```cpp
ListNode* mergeTwoLists(ListNode* list1, ListNode* list2) {
    if (list1 == nullptr) return list2;
    if (list2 == nullptr) return list1;
    if (list1->val <= list2->val) {
        list1->next = mergeTwoLists(list1->next, list2);
        return list1;
    }
    list2->next = mergeTwoLists(list1, list2->next);
    return list2;
}
```

Same $O(n+m)$ time. Space is $O(n+m)$ call-stack frames — one per
attached node. Collect-then-sort is worse on both axes and hides the
pointer lesson.

**Recommendation:** learn the iterative dummy-node version first. It
uses $O(1)$ extra memory and is the same three-move pattern as every
other "build a list from existing nodes" problem. Recursion is a clean
second write, not the one to memorize first.

## Mistakes I made

- Special-casing the first node instead of using a dummy.
- Forgetting `tail = tail->next` after an attach, so later nodes
  overwrite the same `next` and the prefix is lost.
- Advancing `tail` after the leftover splice. Harmless here because we
  return immediately, but it hides the fact that the splice is one
  pointer write, not another loop.
- Writing `tail->next->val = list1->val` and allocating new nodes.
  That copies values. The problem wants the original objects.
- Using `<` vs `<=` as if it mattered for correctness. Either is a
  valid stable-enough merge; pick one and keep it.
- Returning `dummy` instead of `dummy.next`.

## Pattern recognition

Dummy head plus tail is the linked-list version of "append to a
builder." The picture to keep:

```
          processed                   unprocessed

dummy → [ merged nodes ] → tail       list1 → ...
                              \
                               next candidate

                                      list2 → ...
```

`list1` and `list2` search; `tail` builds. Same skeleton as
"merge k lists" (a heap picks the next node) and "partition list"
(two dummy tails, one per bucket).

## Related problems

- Merge k Sorted Lists — this loop, with a heap of current heads
- Sort List — split, recurse, then this merge
- Merge Two Sorted Arrays — the same two-finger idea without pointers
- Partition List / Odd Even Linked List — dummy + tail, different
  attach rule
- Reorder List — reverse the second half, then this zip
- Add Two Numbers — dummy + tail, attaching newly created nodes

## Real-world or engineering connection

Intrusive lists (kernel wait queues, allocator free lists, packet
chains) are merged by retargeting `next`, not by moving payloads. A
dummy head is how those codebases keep "insert at front" from having
a separate empty-list path. Losing the leftover splice is the
linked-list equivalent of dropping a remainder range after a
two-pointer merge.

## Final takeaways

1. The next node is the smaller of the two remaining heads.
2. Dummy + `tail` removes the "is the result empty?" branch.
3. Three moves: attach, advance source, advance `tail`.
4. When one list dies, splice the other in $O(1)$. Do not walk it.
5. Return `dummy.next`.


## Reattempt log

| Date | Mastery | Notes |
|---|---|---|
| 2026-10-06 | 1 | First write-up: iterative C++ dummy-head merge, leftover splice |
